# Anthropic extraction workflow

This template builds a small corpus and extracts structured records with Anthropic text and vision profiles. Set `ANTHROPIC_API_KEY` before starting Jupyter and choose a Messages API model available to your account.

In [ ]:
%env PM_DB=papers.db
%env PM_QUERY=lithium solid electrolyte
%env PM_RECIPE=sse
%env PM_MODEL=YOUR_ANTHROPIC_MODEL
%env PM_OUTPUT=temp_anthropic_materials.csv
%env PM_FINAL=anthropic_materials.csv

## Configure model profiles

Use separate identifiers if the chosen text model does not accept images. `pm_model_status` shows the effective provider, endpoint, and capabilities without printing the secret.

In [ ]:
%%bash
set -euo pipefail
test "$PM_MODEL" != "YOUR_ANTHROPIC_MODEL"
pm_model_config text --provider anthropic --model "$PM_MODEL"
pm_model_config vision --provider anthropic --model "$PM_MODEL"
pm_model_status

## Build the corpus

Search and download credentials are independent of the Anthropic key. This example uses OpenAlex discovery and every configured download source.

In [ ]:
%%bash
set -euo pipefail
pm_search "$PM_QUERY" "$PM_DB" --source openalex --count 25
pm_download "$PM_DB" --format both
pm_corpus_stats "$PM_DB"

## Extract and store records

Begin with five papers. Review the intermediate CSV before increasing the count or using `--force` for a deliberate rerun.

In [ ]:
%%bash
set -euo pipefail
pm_scrape "$PM_DB" "$PM_RECIPE" --mode text-images --image-context paper-text --count 5 --output "$PM_OUTPUT"
pm_store "$PM_DB" "$PM_OUTPUT" "$PM_FINAL" "$PM_RECIPE" --assume-yes
pm_status "$PM_DB"